[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/01_baseline_eval.ipynb)

# Notebook 1 — Baseline Evaluation

Measure LFM2.5-1.2B-Thinking accuracy on StepGame **before** fine-tuning.

In [ ]:
import os, sys
REPO = '/content/spatialft.github.io'
if not os.path.exists(REPO):
    !git clone https://github.com/spatialft/spatialft.github.io.git {REPO}
os.chdir(f'{REPO}/notebooks')
if REPO not in sys.path:
    sys.path.insert(0, REPO)


In [ ]:
# Run once in Colab
# !pip install -r ../requirements.txt

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from src.dataset import load_stepgame, format_prompt
from src.eval import evaluate, save_results


In [ ]:
MODEL_ID = 'LiquidAI/LFM2.5-1.2B-Thinking'
EVAL_PATH = '../data/eval/stepgame_eval.json'
OUT_PATH  = '../results/baseline/predictions.json'
MAX_NEW_TOKENS = 512  # Thinking model needs room for <think> trace
BATCH_SIZE = 8

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

In [ ]:
examples = load_stepgame(EVAL_PATH)
print(f'Loaded {len(examples)} eval examples')
print('Sample:', examples[0])

In [ ]:
predictions = []

for i in tqdm(range(0, len(examples), BATCH_SIZE)):
    batch = examples[i : i + BATCH_SIZE]
    prompts = [format_prompt(ex['story'], ex['question']) for ex in batch]

    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    for ex, output in zip(batch, outputs):
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        predictions.append({
            'story': ex['story'],
            'question': ex['question'],
            'answer': ex['answer'],
            'prediction': generated,
            'k': ex.get('k'),
        })

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(predictions, f, indent=2)


In [ ]:
results = evaluate(predictions)
save_results(results, '../results/baseline/scores.json')

print(f"Overall accuracy: {results['accuracy']:.3f}")
for k, v in results.items():
    if k.startswith('accuracy_k'):
        print(f"  {k}: {v:.3f}")